Database Connection

In [2]:
import os

import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(find_dotenv())

def connect_to_db():
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    database = os.getenv("DB_NAME")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")

    if not all([database, user, password]):
        print("Σφάλμα: Δεν βρέθηκαν οι απαραίτητες μεταβλητές στο .env!")
        return None

    try:
        connection_uri = f"postgresql://{user}:{password}@{host}:{port}/{database}"
        engine = create_engine(connection_uri)
        with engine.connect() as connection:
            print(f"Επιτυχής σύνδεση στη βάση '{database}' στο host '{host}'!")
        return engine

    except SQLAlchemyError as e:
        print(f"Σφάλμα κατά τη σύνδεση στη ΒΔ: {e}")
        return None


print("Εκκίνηση σύνδεσης με τη βάση...")
engine = connect_to_db()

if engine is None:
    print("Τερματισμός προγράμματος λόγω αποτυχίας σύνδεσης.")
    exit(1)

Εκκίνηση σύνδεσης με τη βάση...
Επιτυχής σύνδεση στη βάση 'thesis_db' στο host 'dell-micro'!


Import data to dataframe

In [3]:
import pandas as pd

query = "SELECT match_id, x_loc, y_loc, distance_to_goal, angle_to_goal, is_goal FROM understat_shots_normalized"
us_shots = pd.read_sql(query, con=engine)


query = "SELECT match_id, x_loc, y_loc, distance_to_goal, angle_to_goal, is_goal FROM statsbomb_shots_normalized"
sb_shots = pd.read_sql(query, con=engine)

Remove overlapping matches

In [4]:
query = "SELECT understat_match_id FROM mapping_overlapping_matches"
overlapping_matches = pd.read_sql_query(query, con=engine)
overlapping_matches.head()

,understat_match_id
0,923
1,662
2,727
3,890
4,644


In [5]:
print(f"Αρχικές εγγραφές us_shots: {len(us_shots)}")
overlapping_matches_list = overlapping_matches['understat_match_id'].unique()

# Αφαίρεση overlapping matches από το us_shots
us_shots = us_shots[~us_shots['match_id'].isin(overlapping_matches_list)].reset_index(drop=True)

print(f"Overlapping matches: {len(overlapping_matches_list)}")
print(f"Εγγραφές μετά την αφαίρεση: {len(us_shots)}")

Αρχικές εγγραφές us_shots: 605446
Overlapping matches: 1576
Εγγραφές μετά την αφαίρεση: 566725


Drop 'match_id' from DataFrames

In [6]:
us_shots.drop(['match_id'], axis=1, inplace=True)
sb_shots.drop(['match_id'], axis=1, inplace=True)

print(f"{list(us_shots)} και {list(sb_shots)}")

['x_loc', 'y_loc', 'distance_to_goal', 'angle_to_goal', 'is_goal'] και ['x_loc', 'y_loc', 'distance_to_goal', 'angle_to_goal', 'is_goal']


Merge 2 dataframes

In [7]:
shots = pd.concat([us_shots, sb_shots], ignore_index=True)
print(f"Μέγεθος Understat: {len(us_shots)}")
print(f"Μέγεθος StatsBomb: {len(sb_shots)}")
print(f"Μέγεθος Συνολικό : {len(shots)}")

Μέγεθος Understat: 566725
Μέγεθος StatsBomb: 55857
Μέγεθος Συνολικό : 622582


In [8]:
shots.dtypes

x_loc               float64
y_loc               float64
distance_to_goal    float64
angle_to_goal       float64
is_goal               int64
dtype: object

In [9]:
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss
import numpy as np

all_results = []

def evaluate_model(y_true, y_pred, model_name="Model"):
    loss = log_loss(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    brier = brier_score_loss(y_true, y_pred)
    actual_goals = int(np.sum(y_true))
    expected_goals = float(np.sum(y_pred))

    print(f"╔{'═'*45}╗")
    print(f"  Αποτελέσματα για: {model_name}")
    print(f"╠{'═'*45}╣")
    print(f"  Log Loss:        {loss:.4f}")
    print(f"  ROC AUC:         {auc:.4f}")
    print(f"  Brier Score:     {brier:.4f}")
    print(f"  Actual Goals:    {actual_goals}")
    print(f"  Expected Goals:  {expected_goals:.1f}")
    print(f"╚{'═'*45}╝\n")

    return {
        "Model": model_name,
        "Log Loss": round(loss, 4),
        "ROC AUC": round(auc, 4),
        "Brier Score": round(brier, 4),
        "Actual Goals": actual_goals,
        "Expected Goals": round(expected_goals, 1)
    }

In [10]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold

X = shots.drop(columns=['is_goal'])
y = shots['is_goal']

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
skf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

BL5.1.1.1 — Default Params + Imbalance: None + Calibration: None

In [11]:
bl5_1_1_1 = XGBClassifier(
    device='cuda',
    tree_method='hist',
    eval_metric='logloss',
    verbosity=0,
    random_state=42
)

predictions_bl5_1_1_1 = cross_val_predict(
    bl5_1_1_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_1_1_1, model_name="BL5.1.1.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.1.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.2793
  ROC AUC:         0.7468
  Brier Score:     0.0787
  Actual Goals:    60584
  Expected Goals:  60595.2
╚═════════════════════════════════════════════╝



BL5.1.1.2 — Default Params + Imbalance: None + Calibration: Platt Scaling

In [12]:
from sklearn.linear_model import LogisticRegression

P_raw = predictions_bl5_1_1_1.reshape(-1, 1)

# Χρήση C=np.inf αντί για C=np.inf
platt_calibrator = LogisticRegression(C=np.inf, solver='lbfgs', random_state=42)

predictions_bl5_1_1_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_1_1_2, model_name="BL5.1.1.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.1.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2842
  ROC AUC:         0.7468
  Brier Score:     0.0795
  Actual Goals:    60584
  Expected Goals:  60588.2
╚═════════════════════════════════════════════╝



BL5.1.1.3 — Default Params + Imbalance: None + Calibration: QIsotonic Regression

In [13]:
from sklearn.isotonic import IsotonicRegression

iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl5_1_1_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl5_1_1_3, model_name="BL5.1.1.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.1.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2793
  ROC AUC:         0.7462
  Brier Score:     0.0787
  Actual Goals:    60584
  Expected Goals:  60582.2
╚═════════════════════════════════════════════╝



BL5.1.2.1 — Default Params + Imbalance: Class Weight + Calibration: None

In [14]:
scale_pos_weight_value = (y == 0).sum() / (y == 1).sum()

bl5_1_2_1 = XGBClassifier(
    scale_pos_weight=scale_pos_weight_value,
    device='cuda',
    tree_method='hist',
    eval_metric='logloss',
    verbosity=0,
    random_state=42
)

predictions_bl5_1_2_1 = cross_val_predict(
    bl5_1_2_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_1_2_1, model_name="BL5.1.2.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.2.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5784
  ROC AUC:         0.7464
  Brier Score:     0.1988
  Actual Goals:    60584
  Expected Goals:  258528.6
╚═════════════════════════════════════════════╝



BL5.1.2.2 — Default Params + Imbalance: Class Weight + Calibration: Platt Scaling

In [15]:
P_raw = predictions_bl5_1_2_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(C=np.inf, solver='lbfgs', random_state=42)

predictions_bl5_1_2_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_1_2_2, model_name="BL5.1.2.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.2.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2804
  ROC AUC:         0.7464
  Brier Score:     0.0791
  Actual Goals:    60584
  Expected Goals:  60590.5
╚═════════════════════════════════════════════╝



BL5.1.2.3 — Default Params + Imbalance: Class Weight + Calibration: Isotonic Regression

In [16]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl5_1_2_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl5_1_2_3, model_name="BL5.1.2.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.2.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2794
  ROC AUC:         0.7458
  Brier Score:     0.0787
  Actual Goals:    60584
  Expected Goals:  60584.0
╚═════════════════════════════════════════════╝



BL5.1.3.1 — Default Params + Imbalance: Undersampling + Calibration: None

In [17]:
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from imblearn.under_sampling import RandomUnderSampler

bl5_1_3_1 = make_imb_pipeline(
    RandomUnderSampler(random_state=42),
    XGBClassifier(
        device='cuda',
        scale_pos_weight=1,
        tree_method='hist',
        eval_metric='logloss',
        verbosity=0,
        random_state=42
    )
)

predictions_bl5_1_3_1 = cross_val_predict(
    bl5_1_3_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_1_3_1, model_name="BL5.1.3.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.3.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5901
  ROC AUC:         0.7456
  Brier Score:     0.2029
  Actual Goals:    60584
  Expected Goals:  262866.9
╚═════════════════════════════════════════════╝



BL5.1.3.2 — Default Params + Imbalance: Undersampling + Calibration: Platt Scaling

In [18]:
P_raw = predictions_bl5_1_3_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(C=np.inf, solver='lbfgs', random_state=42)

predictions_bl5_1_3_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_1_3_2, model_name="BL5.1.3.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.3.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2808
  ROC AUC:         0.7456
  Brier Score:     0.0793
  Actual Goals:    60584
  Expected Goals:  60592.5
╚═════════════════════════════════════════════╝



BL5.1.3.3 — Default Params + Imbalance: Undersampling + Calibration: Isotonic Regression

In [19]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl5_1_3_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl5_1_3_3, model_name="BL5.1.3.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.3.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2799
  ROC AUC:         0.7451
  Brier Score:     0.0789
  Actual Goals:    60584
  Expected Goals:  60583.2
╚═════════════════════════════════════════════╝



BL5.1.4.1 — Default Params + Imbalance: Oversampling + Calibration: None

In [20]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE

bl5_1_4_1 = make_imb_pipeline(
    StandardScaler(),
    BorderlineSMOTE(random_state=42),
    XGBClassifier(
        scale_pos_weight=1,
        device='cuda',
        tree_method='hist',
        eval_metric='logloss',
        verbosity=0,
        random_state=42
    )
)

predictions_bl5_1_4_1 = cross_val_predict(
    bl5_1_4_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_1_4_1, model_name="BL5.1.4.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.4.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5702
  ROC AUC:         0.7412
  Brier Score:     0.1972
  Actual Goals:    60584
  Expected Goals:  251566.5
╚═════════════════════════════════════════════╝



BL5.1.4.2 — Default Params + Imbalance: Oversampling + Calibration: Platt Scaling

In [21]:
P_raw = predictions_bl5_1_4_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(C=np.inf, solver='lbfgs', random_state=42)

predictions_bl5_1_4_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_1_4_2, model_name="BL5.1.4.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.4.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2837
  ROC AUC:         0.7412
  Brier Score:     0.0803
  Actual Goals:    60584
  Expected Goals:  60558.3
╚═════════════════════════════════════════════╝



BL5.1.4.3 — Default Params + Imbalance: Oversampling + Calibration: Isotonic Regression

In [22]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl5_1_4_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl5_1_4_3, model_name="BL5.1.4.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.1.4.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2823
  ROC AUC:         0.7407
  Brier Score:     0.0797
  Actual Goals:    60584
  Expected Goals:  60580.7
╚═════════════════════════════════════════════╝



BL5.2.1.1 — HyperParams + Imbalance: None + Calibration: None

In [23]:
import optuna
from sklearn.model_selection import cross_val_predict, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Coarse Search space
def objective_bt(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 400, 1000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),

        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),

        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        'scale_pos_weight': 1,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'logloss',
        'random_state': 42,
        'verbosity': 0
    }

    model = XGBClassifier(**params)

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()

sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 91. Best value: 0.278484: 100%|██████████| 100/100 [22:08<00:00, 13.29s/it]


Βέλτιστες Παράμετροι: {'n_estimators': 850, 'learning_rate': 0.047903862184695264, 'max_depth': 4, 'min_child_weight': 1, 'gamma': 2.5995484815732837e-08, 'subsample': 0.9997934144659133, 'colsample_bytree': 0.9859915634947207, 'reg_alpha': 5.420813126797977, 'reg_lambda': 0.0003023478689713925}
Καλύτερο CV Log Loss: 0.27848


In [ ]:
best_params = {
    'n_estimators': 850,
    'learning_rate': 0.047903862184695264,
    'max_depth': 4,
    'min_child_weight': 1,
    'gamma': 2.5995484815732837e-08,
    'subsample': 0.9997934144659133,
    'colsample_bytree': 0.9859915634947207,
    'reg_alpha': 5.420813126797977,
    'reg_lambda': 0.0003023478689713925
}

In [27]:
# Refined Search space
def objective_bt(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 1600, 2400, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.15, log=True),

        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'min_child_weight': trial.suggest_float('min_child_weight', 3.0, 6.0),
        'gamma': trial.suggest_float('gamma', 1e-4, 0.05, log=True),

        'subsample': trial.suggest_float('subsample', 0.95, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.90, 1.0),

        'reg_alpha': trial.suggest_float('reg_alpha', 5.0, 12.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 15.0, log=True),

        'scale_pos_weight': 1,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'logloss',
        'random_state': 42,
        'verbosity': 0
    }

    model = XGBClassifier(**params)

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()

sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 99. Best value: 0.278313: 100%|██████████| 100/100 [37:18<00:00, 22.38s/it]


Βέλτιστες Παράμετροι: {'n_estimators': 2300, 'learning_rate': 0.1423323211348054, 'max_depth': 2, 'min_child_weight': 3.947639917268321, 'gamma': 0.027652019074191066, 'subsample': 0.9946306837850198, 'colsample_bytree': 0.9079966525759154, 'reg_alpha': 8.966158404561872, 'reg_lambda': 0.5883083150532353}
Καλύτερο CV Log Loss: 0.27831


In [ ]:
# Στο 4ο refined search που έκανα, το n_estimators συνεχίζει να ταβανιάζει, όμως το κέρδος στο log_loss είναι στο 4ο δεκαδικό ψηφίο
# Και επειδή σε όλα τα runs διάλεγε max_depth = 2 (δηλαδή 4 φύλλα), σημαίνει ότι κάθε μεμονωμένο δέντρο δεν έχει χωρητικότητα γι' αυτό και ζητάει συνεχώς περισσότερα δέντρα, κάτι το οποίο μετά απο περίπου 1800 δέντρα, παρατηρείται πολύ μικρή μείωση στο log loss, αλλά πολύ μεγαλύτερο υπολογιστικό κόστος
best_params = {
    'n_estimators': 2300,
    'learning_rate': 0.1423323211348054,
    'max_depth': 2,
    'min_child_weight': 3.947639917268321,
    'gamma': 0.027652019074191066,
    'subsample': 0.9946306837850198,
    'colsample_bytree': 0.9079966525759154,
    'reg_alpha': 8.966158404561872,
    'reg_lambda': 0.5883083150532353
}

In [29]:
best_model_bl5_2_1_1 = XGBClassifier(
    **study_bt.best_params,
    scale_pos_weight=1,
    device='cuda',
    tree_method='hist',
    eval_metric='logloss',
    verbosity=0,
    random_state=42
)

predictions_bl5_2_1_1 = cross_val_predict(
    best_model_bl5_2_1_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_2_1_1, model_name="BL5.2.1.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.1.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.2783
  ROC AUC:         0.7498
  Brier Score:     0.0785
  Actual Goals:    60584
  Expected Goals:  60581.4
╚═════════════════════════════════════════════╝



BL5.2.1.2 — HyperParams + Imbalance: None + Calibration: Platt Scaling

In [34]:
P_raw = predictions_bl5_2_1_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(C=np.inf, solver='lbfgs', random_state=42)

predictions_bl5_2_1_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_2_1_2, model_name="BL5.2.1.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.1.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2837
  ROC AUC:         0.7498
  Brier Score:     0.0794
  Actual Goals:    60584
  Expected Goals:  60584.6
╚═════════════════════════════════════════════╝



BL5.2.1.3 — HyperParams + Imbalance: None + Calibration: Isotonic Regression

In [35]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl5_2_1_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl5_2_1_3, model_name="BL5.2.1.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.1.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2784
  ROC AUC:         0.7492
  Brier Score:     0.0786
  Actual Goals:    60584
  Expected Goals:  60585.6
╚═════════════════════════════════════════════╝



BL5.2.2.1 — HyperParams + Imbalance: Class Weight + Calibration: None

In [36]:
# Coarse Search space
def objective_bt(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 400, 1000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),

        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),

        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        'scale_pos_weight': scale_pos_weight_value,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'logloss',
        'random_state': 42,
        'verbosity': 0
    }

    model = XGBClassifier(**params)

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()

sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 95. Best value: 0.531794: 100%|██████████| 100/100 [1:25:12<00:00, 51.13s/it]


Βέλτιστες Παράμετροι: {'n_estimators': 850, 'learning_rate': 0.04310598188104171, 'max_depth': 12, 'min_child_weight': 1, 'gamma': 1.488690266166408e-07, 'subsample': 0.9122281421849576, 'colsample_bytree': 0.8422368804154138, 'reg_alpha': 0.006041054637238319, 'reg_lambda': 8.998572419305798e-07}
Καλύτερο CV Log Loss: 0.53179


In [ ]:
best_params = {
    'n_estimators': 850,
    'learning_rate': 0.04310598188104171,
    'max_depth': 12,
    'min_child_weight': 1,
    'gamma': 1.488690266166408e-07,
    'subsample': 0.9122281421849576,
    'colsample_bytree': 0.8422368804154138,
    'reg_alpha': 0.006041054637238319,
    'reg_lambda': 8.998572419305798e-07
}

In [38]:
# Refined search space
def objective_bt(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 700, 1500, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08, log=True),

        'max_depth': trial.suggest_int('max_depth', 10, 16),
        'min_child_weight': trial.suggest_int('min_child_weight', 0.2, 3.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 0.1, log=True),

        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.95),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 0.5, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 0.05, log=True),

        'scale_pos_weight': scale_pos_weight_value,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'logloss',
        'random_state': 42,
        'verbosity': 0
    }

    model = XGBClassifier(**params)

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")


Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 99. Best value: 0.518843: 100%|██████████| 100/100 [4:23:59<00:00, 158.40s/it] 


Βέλτιστες Παράμετροι: {'n_estimators': 950, 'learning_rate': 0.027194113408793852, 'max_depth': 16, 'min_child_weight': 0, 'gamma': 2.6986857129177733e-06, 'subsample': 0.9928123509734562, 'colsample_bytree': 0.7835959778569426, 'reg_alpha': 0.03644779957881329, 'reg_lambda': 5.9781549544869194e-05}
Καλύτερο CV Log Loss: 0.51884


In [32]:
# δεν κανουμε και άλλο refined search, γιατι οσο max depth θα του διναμε, θα το χρησιμοποιουσε, συνεπώς θα έκανε overfitting
best_params = {
    'n_estimators': 950,
    'learning_rate': 0.027194113408793852,
    'max_depth': 16,
    'min_child_weight': 0,
    'gamma': 2.6986857129177733e-06,
    'subsample': 0.9928123509734562,
    'colsample_bytree': 0.7835959778569426,
    'reg_alpha': 0.03644779957881329,
    'reg_lambda': 5.9781549544869194e-05
}

In [39]:
best_model_bl5_2_2_1 = XGBClassifier(
    **study_bt.best_params,
    scale_pos_weight=scale_pos_weight_value,
    device='cuda',
    tree_method='hist',
    eval_metric='logloss',
    verbosity=0,
    random_state=42
)

predictions_bl5_2_2_1 = cross_val_predict(
    best_model_bl5_2_2_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_2_2_1, model_name="BL5.2.2.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.2.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5236
  ROC AUC:         0.7158
  Brier Score:     0.1801
  Actual Goals:    60584
  Expected Goals:  209144.4
╚═════════════════════════════════════════════╝



BL5.2.2.2 — HyperParams + Imbalance: Class Weight + Calibration: Platt Scaling

In [40]:
P_raw = predictions_bl5_2_2_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(C=np.inf, solver='lbfgs', random_state=42)

predictions_bl5_2_2_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_2_2_2, model_name="BL5.2.2.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.2.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2919
  ROC AUC:         0.7158
  Brier Score:     0.0819
  Actual Goals:    60584
  Expected Goals:  60597.2
╚═════════════════════════════════════════════╝



BL5.2.2.3 — HyperParams + Imbalance: Class Weight + Calibration: Isotonic Regression

In [41]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl5_2_2_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl5_2_2_3, model_name="BL5.2.2.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.2.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2878
  ROC AUC:         0.7152
  Brier Score:     0.0806
  Actual Goals:    60584
  Expected Goals:  60583.5
╚═════════════════════════════════════════════╝



BL5.2.3.1 — HyperParams + Imbalance: Undersampling + Calibration: None

In [42]:
from imblearn.under_sampling import RandomUnderSampler


# Γενικό search space
def objective_bt(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 400, 1000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),

        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),

        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        'scale_pos_weight': 1,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'logloss',
        'random_state': 42,
        'verbosity': 0
    }

    # Δημιουργία Pipeline: Πρώτα Undersampling, μετά XGBoost
    model = make_imb_pipeline(
        RandomUnderSampler(random_state=42),
        XGBClassifier(**params)
    )

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 39. Best value: 0.586449: 100%|██████████| 100/100 [10:34<00:00,  6.34s/it]


Βέλτιστες Παράμετροι: {'n_estimators': 550, 'learning_rate': 0.189626323583718, 'max_depth': 3, 'min_child_weight': 8, 'gamma': 1.4254148796007286e-08, 'subsample': 0.9411734186381815, 'colsample_bytree': 0.6369187774821821, 'reg_alpha': 3.0335232621449433, 'reg_lambda': 0.07997620125951165}
Καλύτερο CV Log Loss: 0.58645


In [ ]:
best_params = {
    'n_estimators': 550,
    'learning_rate': 0.189626323583718,
    'max_depth': 3,
    'min_child_weight': 8,
    'gamma': 1.4254148796007286e-08,
    'subsample': 0.9411734186381815,
    'colsample_bytree': 0.6369187774821821,
    'reg_alpha': 3.0335232621449433,
    'reg_lambda': 0.07997620125951165
}

In [47]:
# Refined search space
def objective_bt(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 350, 850, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.35, log=True),

        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 4, 15),
        'gamma': trial.suggest_float('gamma', 1e-8, 0.1, log=True),

        'subsample': trial.suggest_float('subsample', 0.85, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),

        'reg_alpha': trial.suggest_float('reg_alpha', 0.5, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 1.0, log=True),

        'scale_pos_weight': 1,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'logloss',
        'random_state': 42,
        'verbosity': 0
    }

    # Δημιουργία Pipeline: Πρώτα Undersampling, μετά XGBoost
    model = make_imb_pipeline(
        RandomUnderSampler(random_state=42),
        XGBClassifier(**params)
    )

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 60. Best value: 0.586418: 100%|██████████| 100/100 [08:27<00:00,  5.07s/it]


Βέλτιστες Παράμετροι: {'n_estimators': 800, 'learning_rate': 0.08662196470873325, 'max_depth': 3, 'min_child_weight': 5, 'gamma': 0.042775695623160456, 'subsample': 0.9997163719207576, 'colsample_bytree': 0.8294664729387506, 'reg_alpha': 2.7423499466319274, 'reg_lambda': 0.501144844845933}
Καλύτερο CV Log Loss: 0.58642


In [ ]:
best_params = {'n_estimators': 800, 'learning_rate': 0.08662196470873325, 'max_depth': 3, 'min_child_weight': 5, 'gamma': 0.042775695623160456, 'subsample': 0.9997163719207576, 'colsample_bytree': 0.8294664729387506, 'reg_alpha': 2.7423499466319274, 'reg_lambda': 0.501144844845933}

In [48]:
best_model_bl5_2_3_1 = make_imb_pipeline(
    RandomUnderSampler(random_state=42),
    XGBClassifier(
        device='cuda',
        scale_pos_weight=1,
        tree_method='hist',
        eval_metric='logloss',
        verbosity=0,
        random_state=42
    )
)

predictions_bl5_2_3_1 = cross_val_predict(
    best_model_bl5_2_3_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]


res = evaluate_model(y, predictions_bl5_2_3_1, model_name="BL5.2.3.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.3.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5901
  ROC AUC:         0.7456
  Brier Score:     0.2029
  Actual Goals:    60584
  Expected Goals:  262866.9
╚═════════════════════════════════════════════╝



BL5.2.3.2 — HyperParams + Imbalance: Undersampling + Calibration: Platt Scaling

In [49]:
P_raw = predictions_bl5_2_3_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(C=np.inf, solver='lbfgs', random_state=42)

predictions_bl5_2_3_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_2_3_2, model_name="BL5.2.3.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.3.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2808
  ROC AUC:         0.7456
  Brier Score:     0.0793
  Actual Goals:    60584
  Expected Goals:  60592.5
╚═════════════════════════════════════════════╝



BL5.2.3.3 — HyperParams + Imbalance: Undersampling + Calibration: Isotonic Regression

In [50]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl5_2_3_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl5_2_3_3, model_name="BL5.2.3.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.3.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2799
  ROC AUC:         0.7451
  Brier Score:     0.0789
  Actual Goals:    60584
  Expected Goals:  60583.2
╚═════════════════════════════════════════════╝



BL5.2.4.1 — HyperParams + Imbalance: Oversampling + Calibration: None

In [43]:
from imblearn.over_sampling import BorderlineSMOTE

def objective_bt(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 400, 1000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),

        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),

        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        'scale_pos_weight': 1,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'logloss',
        'random_state': 42,
        'verbosity': 0
    }

    # Δημιουργία Pipeline: Πρώτα Oversampling, μετά XGBoost
    model = make_imb_pipeline(
        StandardScaler(),
        BorderlineSMOTE(random_state=42),
        XGBClassifier(**params)
    )

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 82. Best value: 0.520346: 100%|██████████| 100/100 [1:32:32<00:00, 55.52s/it]


Βέλτιστες Παράμετροι: {'n_estimators': 1000, 'learning_rate': 0.05457166937482146, 'max_depth': 12, 'min_child_weight': 7, 'gamma': 3.239648142372886e-08, 'subsample': 0.8779663304575879, 'colsample_bytree': 0.905079189442586, 'reg_alpha': 3.7921597352870083e-06, 'reg_lambda': 0.16121592872811585}
Καλύτερο CV Log Loss: 0.52035


In [ ]:
best_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05457166937482146,
    'max_depth': 12,
    'min_child_weight': 7,
    'gamma': 3.239648142372886e-08,
    'subsample': 0.8779663304575879,
    'colsample_bytree': 0.905079189442586,
    'reg_alpha': 3.7921597352870083e-06,
    'reg_lambda': 0.16121592872811585
}

In [51]:
# Refined search space

def objective_bt(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 1000, 1700, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),

        'max_depth': trial.suggest_int('max_depth', 10, 15),
        'min_child_weight': trial.suggest_int('min_child_weight', 4, 12),
        'gamma': trial.suggest_float('gamma', 1e-8, 0.01, log=True),

        'subsample': trial.suggest_float('subsample', 0.8, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.8, 1.0),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-7, 0.01, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.05, 0.8, log=True),

        'scale_pos_weight': 1,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'logloss',
        'random_state': 42,
        'verbosity': 0
    }

    # Δημιουργία Pipeline: Πρώτα Oversampling, μετά XGBoost
    model = make_imb_pipeline(
        BorderlineSMOTE(random_state=42),
        XGBClassifier(**params)
    )

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 76. Best value: 0.516069: 100%|██████████| 100/100 [3:33:12<00:00, 127.93s/it] 


Βέλτιστες Παράμετροι: {'n_estimators': 1300, 'learning_rate': 0.03774060720857189, 'max_depth': 15, 'min_child_weight': 4, 'gamma': 3.553390729457763e-07, 'subsample': 0.9447663798060588, 'colsample_bytree': 0.8264850018276727, 'reg_alpha': 0.002162107745857073, 'reg_lambda': 0.06785921115617678}
Καλύτερο CV Log Loss: 0.51607


In [ ]:
best_params = {
    'n_estimators': 1300,
    'learning_rate': 0.03774060720857189,
    'max_depth': 15,
    'min_child_weight': 4,
    'gamma': 3.553390729457763e-07,
    'subsample': 0.9447663798060588,
    'colsample_bytree': 0.8264850018276727,
    'reg_alpha': 0.002162107745857073,
    'reg_lambda': 0.06785921115617678
}

In [52]:
bl5_2_4_1 = make_imb_pipeline(
    StandardScaler(),
    BorderlineSMOTE(random_state=42),
    XGBClassifier(
        device='cuda',
        scale_pos_weight=1,
        tree_method='hist',
        eval_metric='logloss',
        verbosity=0,
        random_state=42
    )
)

predictions_bl5_2_4_1 = cross_val_predict(
    bl5_2_4_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_2_4_1, model_name="BL5.2.4.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.4.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5702
  ROC AUC:         0.7412
  Brier Score:     0.1972
  Actual Goals:    60584
  Expected Goals:  251566.5
╚═════════════════════════════════════════════╝



BL5.2.4.2 — HyperParams + Imbalance: Oversampling + Calibration: Platt Scaling

In [53]:
P_raw = predictions_bl5_2_4_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(C=np.inf, solver='lbfgs', random_state=42)

predictions_bl5_2_4_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl5_2_4_2, model_name="BL5.2.4.2")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.4.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2837
  ROC AUC:         0.7412
  Brier Score:     0.0803
  Actual Goals:    60584
  Expected Goals:  60558.3
╚═════════════════════════════════════════════╝



BL5.2.4.3 — HyperParams + Imbalance: Oversampling + Calibration: Isotonic Regression

In [54]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl5_2_4_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl5_2_4_3, model_name="BL5.2.4.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL5.2.4.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2823
  ROC AUC:         0.7407
  Brier Score:     0.0797
  Actual Goals:    60584
  Expected Goals:  60580.7
╚═════════════════════════════════════════════╝



Export Results to csv

In [55]:
df_results = pd.DataFrame(all_results)

for col in ['model_name', 'Model', 'model']:
    if col in df_results.columns:
        df_results = df_results.drop_duplicates(subset=[col], keep='last')
        break

df_results.to_csv('BL5-results.csv', index=False, encoding='utf-8-sig')
display(df_results)

,Model,Log Loss,ROC AUC,Brier Score,Actual Goals,Expected Goals
0,BL5.1.1.1,0.2793,0.7468,0.0787,60584,60595.2
1,BL5.1.1.2,0.2842,0.7468,0.0795,60584,60588.2
2,BL5.1.1.3,0.2793,0.7462,0.0787,60584,60582.2
3,BL5.1.2.1,0.5784,0.7464,0.1988,60584,258528.6
4,BL5.1.2.2,0.2804,0.7464,0.0791,60584,60590.5
5,BL5.1.2.3,0.2794,0.7458,0.0787,60584,60584.0
6,BL5.1.3.1,0.5901,0.7456,0.2029,60584,262866.9
7,BL5.1.3.2,0.2808,0.7456,0.0793,60584,60592.5
8,BL5.1.3.3,0.2799,0.7451,0.0789,60584,60583.2
9,BL5.1.4.1,0.5702,0.7412,0.1972,60584,251566.5
